In [ ]:
import itertools
from collections import defaultdict

In [ ]:
N = 4
GOAL = tuple(range(1,17))
ACTIONS = ['u','d','l','r']
DIRS = {'u':-4,'d':4,'l':-1,'r':1}

In [ ]:
def print_board(s):
    for i in range(0,16,4):
        print(*s[i:i+4])
    print()

In [ ]:
def move(state, act):
    s=list(state)
    hole=s.index(16)

    if act=='l' and hole%4==0: return state
    if act=='r' and hole%4==3: return state
    if act=='u' and hole<4: return state
    if act=='d' and hole>11: return state

    nh = hole + DIRS[act]
    s[hole], s[nh] = s[nh], s[hole]
    return tuple(s)

In [ ]:
def project(state, k):
    s=list(state)
    for i in range(16):
        if s[i] > k and s[i] != 16:
            s[i] = 0
    return tuple(s)

In [ ]:
def row1_states():
    out=[]
    for pos in itertools.combinations(range(16),5):
        for vals in itertools.permutations([1,2,3,4,16]):
            s=[0]*16
            for p,v in zip(pos,vals):
                s[p]=v
            out.append(tuple(s))
    return out

In [ ]:
def row2_states():
    out=[]
    for pos in itertools.combinations(range(4,16),5):
        for vals in itertools.permutations([5,6,7,8,16]):
            s=[1,2,3,4] + [0]*12
            for p,v in zip(pos,vals):
                s[p]=v
            out.append(tuple(s))
    return out

In [ ]:
def row3_states():
    out=[]
    for pos in itertools.combinations(range(8,16),9):
        for vals in itertools.permutations([9,10,11,12,13,14,15,16]):
            s=list(range(1,9)) + [0]*8
            for p,v in zip(pos,vals):
                s[p]=v
            out.append(tuple(s))
    return out

In [ ]:
def build_mdp(states, k, base_reward, goal_reward):
    mdp={}
    for s in states:
        mdp[s]={}
        for a in ACTIONS:
            ns = move(s,a)

            corr=0
            for i in range(k):
                if ns[i]==i+1:
                    corr+=1

            if ns[:k]==tuple(range(1,k+1)):
                r = goal_reward
            else:
                r = base_reward + 0.5*corr

            mdp[s][a]=(ns,r)
    return mdp

In [ ]:
def value_iteration(states, mdp, k, gamma=0.9, theta=1e-3):
    V={s:0.0 for s in states}
    goal=tuple(range(1,k+1))

    while True:
        delta=0
        Vnew=V.copy()

        for s in states:
            if s[:k]==goal:
                continue

            best=-1e9
            for a in ACTIONS:
                ns,r = mdp[s][a]
                best=max(best, r + gamma*V[ns])

            delta=max(delta, abs(V[s]-best))
            Vnew[s]=best

        V=Vnew
        if delta < theta:
            break

    policy={}
    for s in states:
        if s[:k]==goal:
            continue

        best=-1e9
        besta=None
        for a in ACTIONS:
            ns,r = mdp[s][a]
            val=r + gamma*V[ns]
            if val>best:
                best=val
                besta=a

        policy[s]=besta

    return policy

In [ ]:
print("Generating states...")
r1 = row1_states()
r2 = row2_states()
r3 = row3_states()

Generating states...


In [ ]:
print("Building MDPs...")
mdp1 = build_mdp(r1,4,-2,60)
mdp2 = build_mdp(r2,8,-4,120)
mdp3 = build_mdp(r3,16,-6,250)

Building MDPs...


In [ ]:
print("Solving policies...")
pol1 = value_iteration(r1,mdp1,4)
pol2 = value_iteration(r2,mdp2,8)
pol3 = value_iteration(r3,mdp3,16)

In [ ]:
start = (
    12,6,2,8,
    10,13,9,4,
    14,15,3,11,
    1,5,16,7
)

In [ ]:
s=start
moves=0

print("Initial State")
print_board(s)

while s[:4] != (1,2,3,4):
    a = pol1[project(s,4)]
    s = move(s,a)
    moves+=1
    print(a)
    print_board(s)

print("ROW 1 DONE")

In [ ]:
while s[:8] != (1,2,3,4,5,6,7,8):
    a = pol2[project(s,8)]
    s = move(s,a)
    moves+=1
    print(a)
    print_board(s)

print("ROW 2 DONE")

In [ ]:
while s != GOAL:
    a = pol3[project(s,16)]
    s = move(s,a)
    moves+=1
    print(a)
    print_board(s)

print("PUZZLE DONE IN", moves, "MOVES")